Edition: 10-2026. 

Designed for: the local SQLite `mimiciii.db` used in the original careflowAI.ipynb.

Primary clinical question:

- Among people still hospitalied and alive 48hr after admission, can information available up to that point predict the remaining time until hospital exit (discharge or in-hospital dealth)?

### 1. Import dependencies

In [ ]:
from pathlib import Path
import sqlite3       # queries data
import sys
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import sklearn

from IPython.display import display

RANDOM_STATE = 42
LANDMARK_HOURS = 48
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

### 2. Connect to local SQLite database

The original notebook expects:

`~/Documents/healthcare_project/data/mimiciii.db`

Edit only DB_PATH if the database is elsewhere. Read-only access avoids accidental modification to the source. The `outputs/` directory will be created in your local project folder.

In [ ]:
PROJECT_DIR = Path.home() / "Documents" / "healthcare_project"
DB_PATH = PROJECT_DIR / "data" / "mimiciii.db"
OUTPUT_DIR = PROJECT_DIR / "outputs"

if not DB_PATH.is_file():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}\n"
        "Update DB_PATH above, and then return this cell."
    )

connection = sqlite3.connect(DB_PATH.resolve().as_uri() + "?mode=ro",
                             uri=True)
print("Connected (read-only) to:", DB_PATH)

## Part 1. Understand the database before joining

`PRAGMA table_info(...)` describes a table without loading it into memory.

The relevant MIMIC-III tables are `PATIENTS`, `ADMISSIONS`, `ICUSTAYS`, `LABEVENTS`, and `CHARTEVENTS`; `D_LABITEMS` and `D_ITEMS` explain the measurement identifiers.

`DIAGNOSES_ICD` is inspected but is not used as a predictor in this baseline model because final codes may be assigned after the landmark.

In [ ]:
required_tables = [
    "PATIENTS", "ADMISSIONS", "ICUSTAYS", "LABEVENTS",
    "CHARTEVENTS", "D_LABITEMS", "D_ITEMS"
]

table_names = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    connection
)["name"].str.upper().tolist()

missing_tables = sorted(set(required_tables) - set(table_names))
if missing_tables:
    raise ValueError(f"Required MIMIC-III tables missing: {missing_tables}")

for table in required_tables + (["DIAGNOSES_ICD"] if "DIAGNOSES_ICD" in table_names else []):
    schema = pd.read_sql_query(f'PRAGMA table_info("{table}")', connection)
    print(f"\n{table}: {len(schema)} columns")
    print(", ".join(schema["name"].head(18).tolist()))

print("\nSchema check passed.")

#### JOIN

`PATIENTS` to `ADMISSIONS` on subject_id

`ADMISSIONS` to `ICUSTAYS` on hadm_id

In [ ]:
icu_sql = """
SELECT
    p.subject_id, p.gender,
    a.hadm_id, a.admission_type, a.admittime, a.dischtime,
    i.icustay_id, i.first_careunit, i.intime, i.outtime,
    i.los AS icu_los_days
FROM PATIENTS AS p
INNER JOIN ADMISSIONS AS a ON p.subject_id = a.subject_id
INNER JOIN ICUSTAYS AS i
    ON a.hadm_id = i.hadm_id AND a.subject_id = i.subject_id
"""
icu_stay_df = pd.read_sql_query(icu_sql, connection)

print("ICU-stay rows:", len(icu_stay_df))
print("Unique ICU stays:", icu_stay_df["icustay_id"].nunique())
print("Unique hospital admissions:", icu_stay_df["hadm_id"].nunique())
print("Unique patients:", icu_stay_df["subject_id"].nunique())
assert icu_stay_df["icustay_id"].is_unique, "Join unexpectedly duplicated ICU stays."
display(icu_stay_df.head())

#### ICU utilization (descriptive)

- `CCU`: Coronary/Cardiac Care Unit
- `CSRU`: Cardiac Surgery Recovery Unit
- `MICU`: Medical Intensive Care Unit
- `SICU`: Surgical Intensive Care Unit
- `TSICU`: Trauma Surgical Intensive Care Unit

In [ ]:
icu_utilization = (
    icu_stay_df.groupby("first_careunit", dropna=False)
    .agg(n_icu_stays=("icustay_id", "size"),
         mean_icu_los=("icu_los_days", "mean"),
         median_icu_los=("icu_los_days", "median"))
    .reset_index()
)
display(icu_utilization.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(icu_utilization["first_careunit"].astype(str), icu_utilization["mean_icu_los"])
ax.set(xlabel="ICU first care unit", ylabel="Mean ICU LOS (days)",
       title="Descriptive: average ICU LOS by unit")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Part 2. Build a one-row-per-admission data table

### Aggregate ICU stays using SQL CTE

*CTE* means Common Table Expression (`WITH ... AS (...)`). Here it collapses multiple ICU-stay rows into one admission-level summary and then performs a `LEFT JOIN` back to all admissions.

`hadm_id` must be unique in `admission_df`.

`total_icu_los_days`, `n_icu_stays`, and `last_icu_outtime` describe the full course of hospitalization. They are for retrospective summaries only; the prediction feature list will explicitly exclude them.

In [ ]:
admission_sql = """
WITH icu_summary AS (
    SELECT subject_id, hadm_id,
           COUNT(*) AS n_icu_stays,
           SUM(los) AS total_icu_los_days,
           MIN(intime) AS first_icu_intime,
           MAX(outtime) AS last_icu_outtime
    FROM ICUSTAYS
    GROUP BY subject_id, hadm_id
)
SELECT a.subject_id, a.hadm_id, p.gender, p.dob,
       a.admission_type, a.admittime, a.dischtime, a.deathtime,
       a.discharge_location, a.hospital_expire_flag,
       i.n_icu_stays, i.total_icu_los_days,
       i.first_icu_intime, i.last_icu_outtime
FROM ADMISSIONS AS a
INNER JOIN PATIENTS AS p ON a.subject_id = p.subject_id
LEFT JOIN icu_summary AS i
    ON a.subject_id = i.subject_id AND a.hadm_id = i.hadm_id
"""
admission_df = pd.read_sql_query(admission_sql, connection)
assert admission_df["hadm_id"].is_unique

for col in ["admittime", "dischtime", "deathtime", "first_icu_intime", "last_icu_outtime"]:
    admission_df[col] = pd.to_datetime(admission_df[col], errors="coerce")

admission_df["hospital_los_days"] = (
    (admission_df["dischtime"] - admission_df["admittime"])
    .dt.total_seconds() / 86400
)

print("All admissions:", len(admission_df))
print("Unique patients:", admission_df["subject_id"].nunique())
print("Missing LOS:", admission_df["hospital_los_days"].isna().sum())
print("LOS <= 0:", (admission_df["hospital_los_days"] <= 0).sum())
display(admission_df[["hadm_id", "subject_id", "hospital_los_days", "n_icu_stays"]].head())

### Fix the age overflow

In MIMIC-III, dates of birth for people older than 89 can be shifted by a very large number of years for de-identification.

`pd.to_datetime(dob)` or subtracting full timestamps can trigger overflows or produce implausible ages.

Solution: Read only the year/month/day string components. Never convert raw DOB to pandas nanosecond timestamps. Calculate approximate age at admission; encode ages above 89 as `90` to respect the MIMIC-III privacy convention. Here `90` means 90 or older, not exact age 90.

In [ ]:
dob_text = admission_df["dob"].astype("string")
adm_year = admission_df["admittime"].dt.year
birth_year = pd.to_numeric(dob_text.str.slice(0, 4), errors="coerce")

# For birthdays later in the admission year, subtract one year.
adm_month_day = admission_df["admittime"].dt.strftime("%m-%d")
birth_month_day = dob_text.str.slice(5, 10)
not_yet_birthday = (adm_month_day < birth_month_day).fillna(False).astype(int)
raw_age = adm_year - birth_year - not_yet_birthday

admission_df["age"] = raw_age.where(raw_age >= 0).clip(upper=90)
print("Age summary (90 = 90+):")
display(admission_df["age"].describe())
print("Missing ages:", admission_df["age"].isna().sum())
print("Top-coded ages (90+):", admission_df["age"].eq(90).sum())

### Descriptive hospital analysis

Mean vs median: Hospital LOS is frequently right-skewed. A few very long stays can increase the mean considerably; the median often better describes a typical stay. Mortality rate is calculated as the mean of a 0/1 flag multiplied by 100.

In [ ]:
hospital_summary = (
    admission_df.groupby("admission_type", dropna=False)
    .agg(n_admissions=("hadm_id", "size"),
         mean_hospital_los=("hospital_los_days", "mean"),
         median_hospital_los=("hospital_los_days", "median"),
         mortality_rate=("hospital_expire_flag", "mean"))
    .reset_index()
)
hospital_summary["mortality_rate_pct"] = 100 * hospital_summary["mortality_rate"]
display(hospital_summary.drop(columns="mortality_rate").round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(admission_df["hospital_los_days"].dropna(), bins=20)
axes[0].set(xlabel="Total hospital LOS (days)", ylabel="Admissions", title="Hospital LOS distribution")
axes[1].bar(hospital_summary["admission_type"].astype(str), hospital_summary["median_hospital_los"])
axes[1].set(xlabel="Admission type", ylabel="Median LOS (days)", title="LOS by admission type")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## Part 3. Exploratory discharge-transition analysis (separate from LOS prediction)

### Inspect actual discharge labels

View the source labels first; then map them to broader categories. Mortality code and destination inconsistencies will be reported.

In [ ]:
destination_summary = (
    admission_df["discharge_location"].fillna("<missing>")
    .value_counts(dropna=False)
    .rename_axis("discharge_location")
    .reset_index(name="n_admissions")
)
destination_summary["percentage"] = 100 * destination_summary["n_admissions"] / len(admission_df)
display(destination_summary.round(1))

### Explicit outcome mapping (only for exploration)

In [ ]:
HOME = {"HOME"}
HOME_HEALTH = {"HOME HEALTH CARE", "HOME WITH HOME IV PROVIDR"}
POST_ACUTE = {"SNF", "REHAB/DISTINCT PART HOSP", "ICF", "LONG TERM CARE HOSPITAL"}

def map_transition(location, expired):
    label = str(location).strip().upper() if pd.notna(location) else ""
    if expired == 1 or label == "DEAD/EXPIRED":
        return "Death"
    if label in HOME:
        return "Home"
    if label in HOME_HEALTH:
        return "Home Health"
    if label in POST_ACUTE:
        return "Post-Acute Facility"
    return "Other"

admission_df["care_transition_category"] = [
    map_transition(location, expired)
    for location, expired in zip(admission_df["discharge_location"],
                                 admission_df["hospital_expire_flag"])
]

transition_summary = (
    admission_df.groupby("care_transition_category")
    .agg(n_admissions=("hadm_id", "size"),
         median_los=("hospital_los_days", "median"),
         mean_los=("hospital_los_days", "mean"))
    .reset_index()
)
transition_summary["percentage"] = 100 * transition_summary["n_admissions"] / len(admission_df)
display(transition_summary.round(2))

mismatch_mask = (
    admission_df["discharge_location"].eq("DEAD/EXPIRED")
    != admission_df["hospital_expire_flag"].eq(1)
)
print("Mortality-label mismatches to review:", int(mismatch_mask.sum()))

### Classification - check the class balance before fitting models.

In [ ]:
service_df = admission_df.loc[
    admission_df["hospital_expire_flag"].eq(0)
    & admission_df["care_transition_category"].isin(["Home", "Home Health", "Post-Acute Facility"])
].copy()
service_df["needs_post_discharge_services"] = (
    service_df["care_transition_category"].isin(["Home Health", "Post-Acute Facility"])
    .astype(int)
)

class_count = service_df["needs_post_discharge_services"].value_counts().sort_index()
print("Discharge-service study admissions:", len(service_df))
print("Outcome 0 = Home; outcome 1 = Services")
print(class_count.to_string())
if len(service_df):
    print("Majority-class baseline accuracy:", round(class_count.max() / len(service_df), 3))

### Univariate comparison: Mann–Whitney U (optional EDA)

Question: Do age distributions differ between the discharge groups? Mann–Whitney U compares ranks; it is not automatically a test of median difference unless distributional assumptions hold. A non-significant p-value is not evidence that groups are identical, especially with a small sample.

In [ ]:
from scipy.stats import mannwhitneyu

home_age = service_df.loc[service_df["needs_post_discharge_services"].eq(0), "age"].dropna()
services_age = service_df.loc[service_df["needs_post_discharge_services"].eq(1), "age"].dropna()

if len(home_age) and len(services_age):
    result = mannwhitneyu(home_age, services_age, alternative="two-sided")
    print("Home admissions:", len(home_age), "| Services admissions:", len(services_age))
    print(f"Mann–Whitney U = {result.statistic:.2f}; p = {result.pvalue:.4f}")
else:
    print("One group has no age data; skip the rank test.")

## Part 4. Define the 48-hour prediction landmark

### The fundamental research design

Time zero: hospital admission (`admittime`). Prediction/landmark time: admission + 48 hours.

Eligibility: the patient is still hospitalized and alive at 48 hours. `dischtime > landmark`; if a recorded `deathtime` exists, `deathtime > landmark`.

Target: `remaining_los_days = (dischtime - prediction_time) / 1 day`. It measures time *until hospital exit* (discharge or death). This is not the same estimand as 'remaining time to live discharge'. Death as a competing event would require a separate approach.

In [ ]:
landmark_df = admission_df.copy()
landmark_df["prediction_time"] = (
    landmark_df["admittime"] + pd.Timedelta(hours=LANDMARK_HOURS)
)

eligible_mask = (
    (landmark_df["dischtime"] > landmark_df["prediction_time"])
    & (landmark_df["deathtime"].isna()
       | (landmark_df["deathtime"] > landmark_df["prediction_time"]))
)
landmark_df = landmark_df.loc[eligible_mask].copy()

landmark_df["remaining_los_days"] = (
    (landmark_df["dischtime"] - landmark_df["prediction_time"])
    .dt.total_seconds() / 86400
)

# Valid status known at the landmark; do not use eventual ICU stay length.
landmark_df["icu_by_48h"] = np.where(
    landmark_df["first_icu_intime"].notna()
    & (landmark_df["first_icu_intime"] < landmark_df["prediction_time"]),
    "Yes", "No"
)

assert landmark_df["hadm_id"].is_unique
assert landmark_df["remaining_los_days"].notna().all()
assert landmark_df["remaining_los_days"].gt(0).all()

print("48-hour eligible admissions:", len(landmark_df))
print("Unique patients:", landmark_df["subject_id"].nunique())
print("Eventually died during admission:", landmark_df["hospital_expire_flag"].eq(1).sum())
print("First ICU admission before landmark:", landmark_df["icu_by_48h"].eq("Yes").sum())
display(landmark_df["remaining_los_days"].describe().round(2))

### Check the label's distribution

For LOS regression, MAE (mean absolute error) will be our primary metric. If the target is severely skewed, many large errors can come from long-stay outliers. We will examine the distribution before modeling, rather than choosing transformations blindly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(landmark_df["remaining_los_days"], bins=20)
axes[0].set(xlabel="Remaining LOS (days)", ylabel="Admissions", title="Target distribution")
axes[1].boxplot(landmark_df["remaining_los_days"].dropna(), vert=True)
axes[1].set(ylabel="Remaining LOS (days)", title="Long-stay outliers")
plt.tight_layout()
plt.show()

## Part 5. Clinical feature engineering: first 48 hours only

Measurement time rule: `admittime <= charttime < prediction_time`.

### Confirm laboratory measurement codes

In [ ]:
lab_itemids = [50912, 51006, 50931, 51222, 51301]
lab_dictionary = pd.read_sql_query(
    "SELECT itemid, label, fluid, category FROM D_LABITEMS WHERE itemid IN (50912,51006,50931,51222,51301) ORDER BY itemid",
    connection
)
display(lab_dictionary)
missing_lab_ids = sorted(set(lab_itemids) - set(lab_dictionary["itemid"].tolist()))
if missing_lab_ids:
    print("WARNING — missing lab dictionary codes:", missing_lab_ids)

### Define reusable SQL landmark eligibility

The lab and vital SQL queries should use the same temporal cohort rule as Python. This reusable CTE prevents accidental drift between separate extraction queries. For an index-friendly SQL comparison we use ISO-style SQLite datetime strings, as found in MIMIC-III; if your source timestamps have a different format, normalize them first.

In [ ]:
ELIGIBLE_ADMISSIONS_CTE = """
eligible_admissions AS (
    SELECT a.hadm_id, a.subject_id, a.admittime,
           datetime(a.admittime, '+48 hours') AS prediction_time
    FROM ADMISSIONS AS a
    WHERE a.dischtime > datetime(a.admittime, '+48 hours')
      AND (a.deathtime IS NULL OR a.deathtime > datetime(a.admittime, '+48 hours'))
)
"""

print("Reusable CTE defined. Check that the 48-hour boundary is exclusive on measurements.")

### Extract lab means by admission

What SQL does here:

1. Limits records to eligible admissions.
2. Keeps records collected between admission and the prediction time.
3. Converts multiple lab-event rows into a single row per admission by conditional AVG.

This matches the original notebook's raw lab means. We have not applied arbitrary lab outlier thresholds; inspect extrema before deciding on a clinically justified preprocessing rule.

In [ ]:
lab_sql = f"""
WITH {ELIGIBLE_ADMISSIONS_CTE},
early_labs AS (
    SELECT a.hadm_id, l.itemid, l.valuenum
    FROM eligible_admissions a
    INNER JOIN LABEVENTS l
        ON l.hadm_id = a.hadm_id AND l.subject_id = a.subject_id
    WHERE l.charttime >= a.admittime
      AND l.charttime < a.prediction_time
      AND l.itemid IN (50912, 51006, 50931, 51222, 51301)
      AND l.valuenum IS NOT NULL
)
SELECT hadm_id,
       AVG(CASE WHEN itemid = 50912 THEN valuenum END) AS creatinine_mean,
       AVG(CASE WHEN itemid = 51006 THEN valuenum END) AS bun_mean,
       AVG(CASE WHEN itemid = 50931 THEN valuenum END) AS glucose_mean,
       AVG(CASE WHEN itemid = 51222 THEN valuenum END) AS hemoglobin_mean,
       AVG(CASE WHEN itemid = 51301 THEN valuenum END) AS wbc_mean
FROM early_labs
GROUP BY hadm_id
"""
early_lab_df = pd.read_sql_query(lab_sql, connection)

assert early_lab_df["hadm_id"].is_unique
assert set(early_lab_df["hadm_id"]).issubset(set(landmark_df["hadm_id"]))
print("Admissions with at least one selected lab event:", len(early_lab_df))
display(early_lab_df.head())

### Check the vital-sign dictionary

In [ ]:
vital_itemids = [211,220045,618,220210,646,220277,455,220179,676,223762,678,223761]
placeholders = ",".join(["?"] * len(vital_itemids))
vital_dictionary = pd.read_sql_query(
    f"SELECT itemid, label, unitname FROM D_ITEMS WHERE itemid IN ({placeholders}) ORDER BY itemid",
    connection, params=vital_itemids
)
display(vital_dictionary)
missing_vital_ids = sorted(set(vital_itemids) - set(vital_dictionary["itemid"].tolist()))
if missing_vital_ids:
    print("WARNING — codes not in the available D_ITEMS dictionary:", missing_vital_ids)

### Extract early vital signs with error screening and temperature conversion

This query retains the SQL chart-event matching to ICUSTAYS from the original file. It is important because ICU charting can contain multiple observations per patient and the icustay_id link establishes provenance.

- COALESCE(c.error, 0) = 0: exclude explicit charting errors.
- CASE WHEN ... THEN ... END: map multiple ITEMIDs into one measurement.
- (F - 32) * 5 / 9: Fahrenheit to Celsius.
- BETWEEN: remove implausible measurement values before averaging.
- COUNT(CASE WHEN ... THEN 1 END): track observation density for temperature/heart rate.

In [ ]:
vital_sql = f"""
WITH {ELIGIBLE_ADMISSIONS_CTE},
chart_48h AS (
    SELECT a.hadm_id, c.itemid, c.valuenum,
           CASE
             WHEN c.itemid IN (676, 223762) THEN c.valuenum
             WHEN c.itemid IN (678, 223761) THEN (c.valuenum - 32.0) * 5.0 / 9.0
           END AS temperature_c
    FROM eligible_admissions a
    INNER JOIN ICUSTAYS i
        ON i.hadm_id = a.hadm_id AND i.subject_id = a.subject_id
    INNER JOIN CHARTEVENTS c
        ON c.icustay_id = i.icustay_id
       AND c.hadm_id = i.hadm_id AND c.subject_id = i.subject_id
    WHERE c.charttime >= a.admittime
      AND c.charttime < a.prediction_time
      AND c.itemid IN (211,220045,618,220210,646,220277,
                       455,220179,676,223762,678,223761)
      AND c.valuenum IS NOT NULL
      AND COALESCE(c.error, 0) = 0
)
SELECT hadm_id,
       AVG(CASE WHEN itemid IN (211,220045) AND valuenum BETWEEN 20 AND 250
                THEN valuenum END) AS heart_rate_mean,
       AVG(CASE WHEN itemid IN (618,220210) AND valuenum BETWEEN 4 AND 80
                THEN valuenum END) AS respiratory_rate_mean,
       AVG(CASE WHEN itemid IN (646,220277) AND valuenum BETWEEN 50 AND 100
                THEN valuenum END) AS spo2_mean,
       AVG(CASE WHEN itemid IN (455,220179) AND valuenum BETWEEN 40 AND 300
                THEN valuenum END) AS sbp_mean,
       AVG(CASE WHEN temperature_c BETWEEN 30 AND 43
                THEN temperature_c END) AS temperature_mean,
       COUNT(CASE WHEN itemid IN (211,220045) AND valuenum BETWEEN 20 AND 250
                  THEN 1 END) AS n_heart_rate,
       COUNT(CASE WHEN temperature_c BETWEEN 30 AND 43
                  THEN 1 END) AS n_temperature
FROM chart_48h
GROUP BY hadm_id
"""
early_vitals_df = pd.read_sql_query(vital_sql, connection)

assert early_vitals_df["hadm_id"].is_unique
assert set(early_vitals_df["hadm_id"]).issubset(set(landmark_df["hadm_id"]))
print("Admissions with at least one selected chart event:", len(early_vitals_df))
display(early_vitals_df.head())

### Combine features using LEFT JOIN logic

The left side is the 48-hour eligible cohort. Every eligible admission must remain in model_df, even if some measurements are missing.

validate='one_to_one' is a valuable data-engineering guard: it fails immediately if a supposed admission-level feature table contains duplicate hadm_ids.

In [ ]:
model_df = (
    landmark_df
    .merge(early_lab_df, on="hadm_id", how="left", validate="one_to_one")
    .merge(early_vitals_df, on="hadm_id", how="left", validate="one_to_one")
)

lab_columns = ["creatinine_mean", "bun_mean", "glucose_mean", "hemoglobin_mean", "wbc_mean"]
vital_columns = ["heart_rate_mean", "respiratory_rate_mean", "spo2_mean", "sbp_mean", "temperature_mean"]
clinical_columns = lab_columns + vital_columns

assert len(model_df) == len(landmark_df), "A join changed the number of admissions."
assert model_df["hadm_id"].is_unique
assert model_df["remaining_los_days"].gt(0).all()

print("Model-ready admissions (before predictor selection):", len(model_df))
print("Number of clinical features:", len(clinical_columns))
display(model_df[["hadm_id", "prediction_time", "remaining_los_days"] + clinical_columns].head())

### Audit missingness and clinical plausibility

Missing completely at random (MCAR), missing at random (MAR), and missing not at random (MNAR) describe different missingness mechanisms. ICU-derived vitals can be structurally missing when a person has not yet entered the ICU. We should report missingness before imputing.

Never impute a feature using the entire dataset before splitting: that would leak the test distribution into training. The model pipeline handles imputation using training data only.

In [ ]:
quality_report = pd.DataFrame({
    "available_n": model_df[clinical_columns].notna().sum(),
    "missing_n": model_df[clinical_columns].isna().sum(),
    "missing_pct": 100 * model_df[clinical_columns].isna().mean(),
    "mean": model_df[clinical_columns].mean(),
    "median": model_df[clinical_columns].median(),
    "min": model_df[clinical_columns].min(),
    "max": model_df[clinical_columns].max(),
})
display(quality_report.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
ordered = quality_report.sort_values("missing_pct")
ax.barh(ordered.index, ordered["missing_pct"])
ax.set(xlabel="Missing admissions (%)", title="Missing first-48-hour clinical features")
ax.set_xlim(0, max(10, float(ordered["missing_pct"].max()) * 1.18))
plt.tight_layout()
plt.show()

### Why are some early vitals missing?

Examine the time from hospital admission to first ICU entry. This variable is used only for retrospective missingness diagnosis here. Do not add a future first-ICU-entry time as a model predictor: future timestamps are unavailable at hour 48.

A patient first admitted to ICU 72 hours after hospital admission can have perfectly good subsequent ICU charting and no ICU vital signs in the first 48 hours.

In [ ]:
timing_audit_df = model_df[["hadm_id", "admittime", "first_icu_intime", "heart_rate_mean"]].copy()
timing_audit_df["hours_to_first_icu"] = (
    (timing_audit_df["first_icu_intime"] - timing_audit_df["admittime"])
    .dt.total_seconds() / 3600
)
timing_audit_df["icu_timing_group"] = pd.cut(
    timing_audit_df["hours_to_first_icu"],
    bins=[-np.inf, 0, 24, 48, np.inf],
    labels=["Before/at admission", "0–24h", "24–48h", "After 48h"]
)
timing_audit_df["icu_timing_group"] = (
    timing_audit_df["icu_timing_group"].cat.add_categories(["No ICU record"])
    .fillna("No ICU record")
)
timing_audit_df["missing_heart_rate"] = timing_audit_df["heart_rate_mean"].isna()

timing_summary = (
    timing_audit_df.groupby("icu_timing_group", observed=True)
    .agg(admissions=("hadm_id", "size"), missing_hr=("missing_heart_rate", "sum"))
    .reset_index()
)
timing_summary["missing_hr_pct"] = 100 * timing_summary["missing_hr"] / timing_summary["admissions"]
display(timing_summary.round(1))

## Part 6. Define prediction-time features, then audit leakage

### Feature

Allowed at 48 hours: age, sex/gender, admission type, whether ICU entry had occurred by 48 hours, and selected first-48-hour labs/vitals. Some source variables still need operational timestamp auditing before real-time use.

Not allowed: full hospital LOS, full ICU LOS, total ICU-stay count, discharge location, in-hospital mortality outcome, eventual care transition, and the timestamps of later events.

Why whitelist? Using X = df.drop('target') is dangerous in EHR data: it accidentally retains other outcomes and future-known columns. Instead explicitly list every permitted predictor.

In [ ]:
numeric_features = ["age"] + clinical_columns
categorical_features = ["gender", "admission_type", "icu_by_48h"]
feature_columns = numeric_features + categorical_features
TARGET = "remaining_los_days"

# Explicitly forbid retrospective/outcome variables from predictor inputs.
forbidden_columns = {
    "hospital_los_days", "total_icu_los_days", "n_icu_stays",
    "dischtime", "deathtime", "discharge_location",
    "hospital_expire_flag", "care_transition_category",
    "needs_post_discharge_services", "last_icu_outtime",
    "first_icu_intime", "hours_to_first_icu", "prediction_time", "dob"
}
assert not forbidden_columns.intersection(feature_columns)
assert set(feature_columns).issubset(model_df.columns)

X = model_df[feature_columns].copy()
y = model_df[TARGET].astype(float).copy()
groups = model_df["subject_id"].copy()

print("Feature columns:", feature_columns)
print("X shape:", X.shape, "| y:", y.shape)
print("Unique patients:", groups.nunique())
assert X.index.equals(y.index) and X.index.equals(groups.index)

### Optional integrity checks you can reuse

These test the expected shape and clinical time rule. They cannot prove data availability at the bedside, which requires additional timestamp metadata.

In [ ]:
assert model_df["hadm_id"].is_unique
assert not y.isna().any()
assert (y > 0).all()
assert (model_df["prediction_time"] - model_df["admittime"]).eq(pd.Timedelta(hours=48)).all()
assert model_df["deathtime"].isna().all() or (
    (model_df["deathtime"].isna())
    | (model_df["deathtime"] > model_df["prediction_time"])
).all()

# SQL query uses [admission, landmark) for lab/chartevent charttime.
assert "l.charttime < a.prediction_time" in lab_sql
assert "c.charttime < a.prediction_time" in vital_sql

print("Shape, outcome, and temporal-boundary checks passed.")

## Part 7. Train/test split — at the patient level

### Prevent admission leakage across train and test

A person may have multiple admissions. Randomly splitting rows could put one patient's earlier admission in training and a later admission in test, causing dependence and overly optimistic performance.

Use GroupShuffleSplit with subject_id as the group. Reserve about 20% of patients for an untouched holdout. Do not use the test set to select predictors or hyperparameters.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate

if groups.nunique() < 8:
    raise ValueError("Too few distinct patients for a meaningful grouped split. Need a larger sample.")

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

assert set(groups_train).isdisjoint(set(groups_test)), "Patient leakage between train and test!"

print("Train admissions:", len(X_train), "| Test admissions:", len(X_test))
print("Train patients:", groups_train.nunique(), "| Test patients:", groups_test.nunique())
print("Patient overlap:", len(set(groups_train).intersection(set(groups_test))))

## Part 8. Fit a baseline and compare simple machine-learning models
### Create a leakage-safe preprocessing pipeline

- Numeric imputation: replace missing values with the training-fold median.
- Missing indicators: mark missing numeric values, including absent early ICU vitals.
- Standardization: important for Ridge regression, less important for trees.
- Categorical features: impute most-frequent category, then one-hot encode (e.g., admission_type = ELECTIVE).
- Pipeline: keeps all learned preprocessing inside cross-validation folds.

Models: median-only baseline, Ridge regression (regularized linear model), and Random Forest (nonlinear tree ensemble). These models are benchmarks, not clinical-grade algorithms.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

def build_pipeline(regressor):
    preprocess = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])
    return Pipeline([("preprocess", preprocess), ("model", regressor)])

candidates = {
    "Median baseline": build_pipeline(DummyRegressor(strategy="median")),
    "Ridge": build_pipeline(Ridge(alpha=10.0)),
    "Random Forest": build_pipeline(RandomForestRegressor(
        n_estimators=150, min_samples_leaf=4,
        random_state=RANDOM_STATE, n_jobs=-1
    )),
}
print("Candidate models:", ", ".join(candidates))

### Patient-grouped cross-validation, using training data only

GroupKFold repeatedly holds out different patients from the training set. In each fold, imputation and scaling are refitted from that fold's training patients.

Lower MAE and RMSE are better. MAE in days is easier to explain to a clinician. RMSE penalizes large mistakes more heavily. Choose the model using mean cross-validated training MAE, not test MAE.

In [ ]:
n_splits = min(5, groups_train.nunique())
cv = GroupKFold(n_splits=n_splits)
cv_rows = []

for name, estimator in candidates.items():
    score = cross_validate(
        estimator, X_train, y_train,
        groups=groups_train,
        cv=cv,
        scoring={"mae": "neg_mean_absolute_error", "mse": "neg_mean_squared_error"},
        error_score="raise",
    )
    cv_rows.append({
        "model": name,
        "cv_mae_days": float((-score["test_mae"]).mean()),
        "cv_mae_sd": float((-score["test_mae"]).std()),
        "cv_rmse_days": float(np.sqrt(-score["test_mse"]).mean()),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("cv_mae_days").reset_index(drop=True)
display(cv_results.round(3))

best_name = cv_results.loc[0, "model"]
print("Selected by TRAINING grouped CV:", best_name)

### Evaluate the selected model once on untouched patients

Holdout testing simulates performance on patients not previously encountered in development. Do not change your model repeatedly based on these numbers; that would indirectly train on your test set.

R²: 1 is perfect; 0 is comparable to predicting the test-set mean in the conventional formula; negative R² is possible and signals poor predictions relative to that reference.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = candidates[best_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

holdout_metrics = {
    "selected_model": best_name,
    "n_test_admissions": len(y_test),
    "n_test_patients": groups_test.nunique(),
    "MAE_days": mean_absolute_error(y_test, y_pred),
    "RMSE_days": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2": r2_score(y_test, y_pred),
}
display(pd.DataFrame([holdout_metrics]).round(3))
print("Negative predicted LOS values:", int((y_pred < 0).sum()))

### Error analysis: prediction versus actual outcome

For the scatterplot, perfect predictions lie on the diagonal. Marked underprediction among patients with very long stays is a common failure mode for LOS models. Remember: these predictions represent time until any hospital exit, not time to live discharge alone.

In [ ]:
prediction_df = model_df.iloc[test_idx][[
    "subject_id", "hadm_id", "remaining_los_days", "hospital_expire_flag"
]].copy()
prediction_df["predicted_remaining_los_days"] = y_pred
prediction_df["absolute_error_days"] = abs(
    prediction_df["remaining_los_days"] - prediction_df["predicted_remaining_los_days"]
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(prediction_df["remaining_los_days"],
           prediction_df["predicted_remaining_los_days"], alpha=0.75)
lims = [min(0, prediction_df[["remaining_los_days", "predicted_remaining_los_days"]].min().min()),
        prediction_df[["remaining_los_days", "predicted_remaining_los_days"]].max().max()]
ax.plot(lims, lims, "--", linewidth=1)
ax.set(xlabel="Actual remaining LOS (days)", ylabel="Predicted remaining LOS (days)",
       title=f"Held-out predictions — {best_name}")
plt.tight_layout()
plt.show()

display(prediction_df.sort_values("absolute_error_days", ascending=False).head(10).round(2))

### Post hoc mortality-stratified error audit

The model is trained to predict time to hospital exit. Death can produce a clinically different LOS process than discharge. Evaluate errors in both groups but do not include the final mortality flag in the features. This subgroup audit is exploratory; counts in a tiny test set will be unstable.

In [ ]:
mortality_error = (
    prediction_df.groupby("hospital_expire_flag", dropna=False)
    .agg(n_admissions=("hadm_id", "size"),
         mae_days=("absolute_error_days", "mean"),
         median_error_days=("absolute_error_days", "median"))
    .reset_index()
)
display(mortality_error.round(2))

### Permutation importance

A predictor has positive permutation importance if randomly shuffling that column makes MAE worse. Negative importance means the shuffled model sometimes performs better by chance. Importance is not causal and is very noisy on a small test cohort.

This is a post hoc test-set description only. Do not repeatedly remove features and refit on the same holdout using these ranks; doing so would contaminate the held-out evaluation.

In [ ]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    best_model, X_test, y_test,
    scoring="neg_mean_absolute_error", n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=1
)
importance_df = (
    pd.DataFrame({
        "feature": feature_columns,
        "mae_increase_days": importance.importances_mean,
        "importance_sd": importance.importances_std,
    })
    .sort_values("mae_increase_days", ascending=False)
)
display(importance_df.round(3))

## Part 9. Save reproducible outputs (locally)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_columns = ["subject_id", "hadm_id", "remaining_los_days"] + feature_columns
model_df[export_columns].to_csv(OUTPUT_DIR / "careflow_48h_model_table.csv", index=False)
cv_results.to_csv(OUTPUT_DIR / "careflow_cv_results.csv", index=False)
pd.DataFrame([holdout_metrics]).to_csv(OUTPUT_DIR / "careflow_holdout_metrics.csv", index=False)
prediction_df.to_csv(OUTPUT_DIR / "careflow_holdout_predictions.csv", index=False)

print("Saved to:", OUTPUT_DIR.resolve())
for name in ["careflow_48h_model_table.csv", "careflow_cv_results.csv",
             "careflow_holdout_metrics.csv", "careflow_holdout_predictions.csv"]:
    print("-", name)